# 10a — XAI định lượng cấp 3 trên DB15K và FB15K-237 (fidelity, validity, sparsity)

Đề, mục 5 (cấp 3): đánh giá giải thích định lượng — fidelity (bỏ hoặc làm nhiễu cạnh/đường đi được dùng làm giải thích thì điểm
hoặc thứ hạng phải thay đổi theo hướng hợp lý), validity và sparsity, bằng chứng cụ thể cho tối thiểu 10 truy vấn.

Notebook này đo các chỉ số đó cho **đúng các giải thích của cấp 2** (notebook 07: E3, học lại nhúng cấu trúc kiểu Kelpie) trên
checkpoint 3 seed đã khóa của nhóm. Không huấn luyện lại, không chọn gì trên test. K\* lấy từ C6 của cấp 2 (chọn trên validation).

| Bước | Nội dung | File kết quả (`0. final/results/<tập>/XAI_L3/`) |
|---|---|---|
| Q0 | Nạp dữ liệu, checkpoint, mô hình (mã giữ nguyên từ notebook 07) | — |
| Q1 | Truy vấn: 60 truy vấn gián tiếp của C9 + 10 truy vấn gián tiếp của bộ 20 (C3); kiểm tra tái lập hạng | `D1_replay_check.csv` |
| Q2 | K\* từ C6 cấp 2 | — |
| Q3 | Với mỗi truy vấn: giải thích ảnh hạng 1 trả về, rồi bỏ / chỉ giữ / làm nhiễu bộ ba giải thích | `D3_fidelity_mimic_per_query.csv` |
| Q4 | Fid+, Fid−, characterization, làm nhiễu; so với K bộ ba ngẫu nhiên và K bộ ba "phổ biến" | `D3_fidelity_mimic_summary.csv` |
| Q5 | Sparsity và đường cong theo K (seed chính) | `D5_sparsity_K_curve.csv` |
| Q6 | Validity phản thực và validity cấu trúc | `D6_validity.csv` |

**Định nghĩa** (S = top-K bộ ba của E3; F(e\*) = mọi bộ ba train của thực thể e\* gắn với ảnh trả về; p = softmax của điểm cuối
α·z(CLIP) + (1−α)·z(đồ thị) trên top-1000):

| Chỉ số | Cách tính | Hướng tốt |
|---|---|---|
| Fid+ (cần thiết) | (p − p₋ₛ) / p khi bỏ S khỏi F(e\*) | ↑ |
| Fid− (đủ) | (p − pₛ) / p khi chỉ giữ S | ↓ |
| Characterization | 2·Fid+·(1 − Fid−) / (Fid+ + 1 − Fid−), sau khi kẹp Fid± về [0, 1] | ↑ |
| Fidelity khi làm nhiễu | (p − pₚₑᵣₜ) / p khi thay đầu kia của mỗi bộ ba trong S bằng thực thể ngẫu nhiên từng đứng cùng vai trò với quan hệ đó | ↑ |
| Sparsity | 1 − \|S\| / \|F(e\*)\| | ↑ (khi fidelity vẫn đạt) |
| Validity phản thực (PN) | % truy vấn mà bỏ S làm ảnh trả về mất hạng 1 | ↑ |
| Đủ để giữ hạng 1 (PS) | % truy vấn mà chỉ giữ S ảnh vẫn hạng 1 | ↑ |
| Validity cấu trúc | (i) mọi bộ ba của S là bộ ba train, không trùng val/test; (ii) % bộ ba của S nằm trên đường đi ≤ 2 bước từ anchor | (i) = 100%, (ii) ↑ |

Fid± chuẩn hóa theo p để so được giữa các truy vấn (p tuyệt đối nhỏ vì softmax trên 1000 ứng viên); bản chưa chuẩn hóa vẫn lưu.
Công thức theo khảo sát XAI cho GNN (Amara và cs., arXiv 2207.12599). Đường cơ sở: K bộ ba ngẫu nhiên từ F(e\*) (5 lần) và K bộ ba
có đầu kia bậc cao nhất.

**Chạy:** Colab GPU (T4 đủ) → Run all. Ước khoảng 1–1,5 giờ cho 2 tập × 3 seed (DB15K chiếm phần lớn vì nhúng 500 chiều);
kết quả từng seed được ghi ngay vào `XAI_L3/_D3_seed*.csv`, nên Colab ngắt thì Run all lại sẽ tiếp tục từ seed chưa xong.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 1, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 2, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 3, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 4, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 5, globals())


## Q1. Truy vấn dùng để đo và kiểm tra tái lập

Đúng 60 truy vấn gián tiếp mà C9 của cấp 2 đã bốc (`SELECTION_SEED = 2026`) và 10 truy vấn gián tiếp của bộ 20 truy vấn (C3).
Truy vấn trực tiếp không đo: ở DB15K điểm đồ thị không tham gia (chỉ CLIP); ở FB15K ablation cho thấy bộ ba không đóng góp,
nên giải thích bằng bộ ba cho truy vấn trực tiếp sẽ là giải thích sau khi có kết quả (đề cấm, mục 2).
Hạng CLIP và hạng sau rerank của từng truy vấn được so với file `C1_test_ranks_seed*.csv` của notebook 07 (hoặc `test_query_ranks.csv` của nhóm).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 7, globals())


## Q2. K\* — số bộ ba trong một giải thích

Dùng lại K\* mà C6 của cấp 2 đã chọn trên validation (K nhỏ nhất trong {1, 2, 3, 5, 8} có trung vị độ đầy đủ ≥ 0,8).
Nếu chưa có file đó thì chạy lại đúng thủ tục C6.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 9, globals())


## Q3. Đo fidelity cho từng giải thích

Lượt 1 giống hệt E3 cấp 2 (bỏ lần lượt từng bộ ba ứng viên) để xếp hạng bộ ba, nên S trùng với giải thích mà notebook 07 hiển thị.
Lượt 2 học lại nhúng cấu trúc của e\* cho mọi biến thể cùng lúc (cùng khởi tạo, cùng mẫu âm, cùng số bước): bỏ S, chỉ giữ S,
làm nhiễu S, bỏ K bộ ba ngẫu nhiên, bỏ K bộ ba phổ biến, chỉ giữ K bộ ba ngẫu nhiên. Chênh lệch điểm so với bản học lại đủ bộ ba
được trừ vào **điểm gốc** của checkpoint, rồi tính lại hạng và p trên đúng hàm xếp hạng của nhóm.
Seed chính (2027) đo thêm mọi K trong {1, 2, 3, 5, 8} để vẽ đường cong (bỏ / chỉ giữ S, 3 lần bỏ ngẫu nhiên); đủ mọi biến thể
(làm nhiễu, bộ ba phổ biến, 5 lần ngẫu nhiên) chỉ đo ở K\*, trên cả 3 seed.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 11, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 12, globals())


## Q4. Fidelity tại K\*: trung bình mỗi seed, rồi trung bình ± độ lệch chuẩn qua 3 seed

`win_rate` = tỉ lệ truy vấn mà bỏ S làm p giảm nhiều hơn trung bình 5 lần bỏ K bộ ba ngẫu nhiên. `wilcoxon_p` = kiểm định Wilcoxon
cặp (một phía) giữa Fid+ của S và Fid+ ngẫu nhiên trên cùng truy vấn.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 14, globals())


## Q5. Sparsity và đường cong theo K (seed chính)

K nhỏ thì giải thích gọn (sparsity cao) nhưng có thể chưa đủ (Fid− cao). Đường cong cho thấy đánh đổi này; K\* là điểm đã khóa trên validation.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 16, globals())


## Q6. Validity

- **Phản thực (PN)**: bỏ S thì ảnh trả về mất hạng 1. So với bỏ K bộ ba ngẫu nhiên (PN ngẫu nhiên).
- **Cấu trúc**: (i) mọi bộ ba của S là bộ ba train và không trùng bộ ba val/test — phải đạt 100%;
  (ii) tỉ lệ bộ ba của S nằm trên đường đi ≤ 2 bước từ anchor tới e\*. Tiêu chí (iii) đúng kiểu thực thể chỉ đo trên MMKG (notebook 10b),
  vì DB15K và FB15K-237 không có kiểu thực thể đi kèm.
- **Theo người**: chấm ở notebook 10b/10d trên MMKG.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('10a_xai_quant_db15k_fb15k', 18, globals())
